In [5]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent   # /mnt/primary/Main
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from loader.clean_stage import *
from loader.stage_config import *

from trino_stack.workload import annotate_metrics_csv_inplace_old


In [7]:

"""
Workload Setup config
"""

instance = "lakehouse-a"
schema = "tpcds"
run_ids = ["20260429-135222Z",
           "20260430-153315Z",
           "20260501-174214Z",
          ]
collection = "tpcds_500_new"


In [6]:


"""
labels raw continuous metrics with the query that was running at each timestamp.  
NOTE: lakehouse.run_workload should have already done this but can be done manually if failed 
"""

RELABEL_METRICS = False

if RELABEL_METRICS:
    for run_id in run_ids:
        run_dir = Path(f"/mnt/lakehouse-raw-results/{schema}/{instance}/{run_id}")
        log_path = run_dir / "workload_log.ndjson"
        for metrics_file in run_dir.glob("*_metrics.csv"):
            print(f"Annotating {metrics_file}")
            annotate_metrics_csv_inplace_old(
                metrics_file,
                log_path,
                pad_ms=0,
                query_id_field="trino_query_id",
                backup=True,
            )

In [8]:

"""
Prepare Raw files for modelling 
"""

parse_results(
    schema=schema,
    instance=instance,
    run_ids=run_ids,
    collection=collection,
    parse_metrics=True,
    parse_profiles=True,
)

2026-05-04 10:45:53,840 [INFO] Results: /mnt/lakehouse-raw-results/tpcds/lakehouse-a
2026-05-04 10:45:53,841 [INFO] Output : /mnt/primary/Main/Parsed_Results/tpcds_500_new/tpcds/lakehouse-a
2026-05-04 10:45:53,841 [INFO] Node glob: trino-worker-*_metrics.csv
2026-05-04 10:45:53,841 [INFO] Coord glob: trino-coord-pod-*_metrics.csv
2026-05-04 10:45:53,842 [INFO] Also stage coord: True
2026-05-04 10:45:53,842 [INFO] Parse metrics: True
2026-05-04 10:45:53,842 [INFO] Parse profiles: True
2026-05-04 10:45:53,843 [INFO] Allow profile-only runs: True
2026-05-04 10:45:53,843 [INFO] Grid points per query: 400
2026-05-04 10:45:55,109 [INFO] Run 20260429-135222Z -> loaded 2500 profile rows
2026-05-04 10:45:55,110 [INFO] Run 20260429-135222Z -> found 4 worker CSVs and 1 coordinator CSVs
2026-05-04 10:45:55,110 [INFO] Run 20260429-135222Z -> staging 4 worker CSVs and 1 coordinator CSVs
/mnt/primary/Main/loader/clean_stage.py:99: DtypeWarning: Columns (7,8) have mixed types. Specify dtype option on 